# Continental founder-spread explorer

A visual synthesis of the genealogy, migration, endogamy, Tasmania-isolation, and genealogy-versus-DNA models.

### Running in Google Colab

Choose **Runtime → Run all** (or run the cells from top to bottom). The first code cell clones/updates this repository and installs the package. Wait until it prints **`Environment ready:`** before the import/dashboard cells run.

**What this can answer:** conditional questions such as *“under these population, migration, mating, barrier, and timing assumptions, how often does ancestry from both founders reach every modeled region?”*

**What it cannot answer:** the historical probability that Adam and Eve existed. Migration and mixing settings remain sensitivity assumptions unless independently constrained.

In [ ]:
import os, sys, subprocess
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_DIR = None
if IN_COLAB:
    REPO_DIR = Path('/content/Evolution-Creation')
    if not (REPO_DIR / '.git').exists():
        subprocess.run(['git','clone','-q','https://github.com/vafaei-ar/Evolution-Creation.git',str(REPO_DIR)],check=True)
    else:
        subprocess.run(['git','-C',str(REPO_DIR),'fetch','-q','origin','main'],check=True)
        subprocess.run(['git','-C',str(REPO_DIR),'checkout','-q','main'],check=True)
        subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard','origin/main'],check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{REPO_DIR}[dev]'],check=True)
else:
    for candidate in (Path.cwd(), Path.cwd().parent):
        if (candidate / 'src' / 'evolution_creation').exists():
            REPO_DIR = candidate
            break

if REPO_DIR is not None:
    src_path = str(REPO_DIR / 'src')
    if src_path not in sys.path:
        sys.path.insert(0, src_path)

print('Environment ready:', REPO_DIR if REPO_DIR is not None else 'using installed Python environment')


In [ ]:
import json
from pathlib import Path
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from evolution_creation.continental_explorer import (
    compare_scenarios,
    diagnose_scenario,
    first_generation_reaching,
    parent_source_matrix_from_offdiag,
    simulate_continental_explorer,
    simulate_continental_stochastic,
)

default_path = (REPO_DIR / 'data' / 'continental_explorer_defaults.json') if REPO_DIR is not None else Path('data/continental_explorer_defaults.json')
if not default_path.exists(): default_path = Path('../data/continental_explorer_defaults.json')
defaults = json.loads(default_path.read_text())
regions = defaults['regions']
coords = np.array([defaults['coordinates'][name] for name in regions], dtype=float)
region_colors = ['#7b2cbf','#2a9d8f','#457b9d','#f4a261','#e76f51','#264653','#d00000']
saved_scenarios = {}


## Controls

Use a preset as a starting point, then edit any value. **Tasmania is a separate node** so a hard Bass Strait barrier can be represented directly.

In [ ]:
# --- high-level scenario controls ---
preset = widgets.Dropdown(options=list(defaults['scenario_presets']),value='Teaching default',description='Preset')
founder_age = widgets.IntSlider(value=11000,min=1000,max=12000,step=100,description='Years ago',continuous_update=False)
generation_interval = widgets.FloatSlider(value=28.0,min=20.0,max=35.0,step=.5,description='Years/gen',continuous_update=False)
founder_region = widgets.Dropdown(options=regions,value='Middle East',description='Origin')
joint_children = widgets.IntSlider(value=2,min=0,max=20,step=1,description='Joint children')
mode = widgets.ToggleButtons(options=[('Deterministic','det'),('Monte Carlo','mc')],value='det',description='Mode')
replicates = widgets.IntSlider(value=int(defaults['stochastic_default']['replicates']),min=25,max=500,step=25,description='Replicates',continuous_update=False)
seed = widgets.IntText(value=int(defaults['stochastic_default']['seed']),description='Seed')
map_metric = widgets.Dropdown(options=[('Both founders','both'),('Either founder','any'),('Mean founder DNA','genetic')],value='both',description='Map')

# --- population controls ---
auto_population = widgets.Checkbox(value=True,description='Auto start population from founder date',indent=False)
population_status = widgets.HTML()
start_boxes, target_boxes = {}, {}
pop_grid = widgets.GridspecLayout(len(regions)+1,3,width='790px')
pop_grid[0,0]=widgets.HTML('<b>Region</b>'); pop_grid[0,1]=widgets.HTML('<b>Start</b>'); pop_grid[0,2]=widgets.HTML('<b>Target</b>')
for i,name in enumerate(regions,start=1):
    pop_grid[i,0]=widgets.HTML(name)
    start_boxes[name]=widgets.IntText(value=int(defaults['population_preset_9000_bce']['values'][name]),layout=widgets.Layout(width='180px'))
    target_boxes[name]=widgets.IntText(value=int(defaults['population_target_2000_ce']['values'][name]),layout=widgets.Layout(width='180px'))
    pop_grid[i,1]=start_boxes[name]; pop_grid[i,2]=target_boxes[name]

# --- migration and mixing controls ---
migration_scale = widgets.FloatSlider(value=1.0,min=0,max=10,step=.1,description='Migration ×',continuous_update=False)
late_contact_age = widgets.IntSlider(value=int(defaults['late_contact_default']['start_years_ago']),min=0,max=3000,step=25,description='Late contact',continuous_update=False)
late_multiplier = widgets.FloatSlider(value=float(defaults['late_contact_default']['external_parent_multiplier']),min=1,max=20,step=.5,description='Late ×',continuous_update=False)
mix_boxes={}
for name,value in zip(regions,defaults['mixing_strength_default']['values']):
    mix_boxes[name]=widgets.FloatSlider(value=float(value),min=0,max=1,step=.05,description=name,continuous_update=False,layout=widgets.Layout(width='450px'))

base_offdiag=np.array(defaults['parent_source_offdiag_default']['values'],dtype=float)
matrix_boxes={}
matrix_grid=widgets.GridspecLayout(len(regions)+1,len(regions)+1,width='1220px')
matrix_grid[0,0]=widgets.HTML('<b>destination ↓ / source →</b>')
for j,name in enumerate(regions,start=1): matrix_grid[0,j]=widgets.HTML(f'<b>{name}</b>')
for i,dest in enumerate(regions,start=1):
    matrix_grid[i,0]=widgets.HTML(f'<b>{dest}</b>')
    for j,source in enumerate(regions,start=1):
        if i==j: matrix_grid[i,j]=widgets.HTML('<i>local auto</i>')
        else:
            box=widgets.FloatText(value=100*base_offdiag[i-1,j-1],step=.001,layout=widgets.Layout(width='105px'))
            matrix_boxes[(i-1,j-1)]=box; matrix_grid[i,j]=box

# --- Tasmania barrier controls ---
tasmania_barrier = widgets.Checkbox(value=True,description='Hard Australia/Oceania ↔ Tasmania barrier before release',indent=False)
permanent_barrier = widgets.Checkbox(value=False,description='Keep Tasmania barrier closed through present',indent=False)
tasmania_release = widgets.IntSlider(value=int(defaults['tasmania_barrier_default']['release_years_ago']),min=0,max=3000,step=10,description='Release y ago',continuous_update=False)


In [ ]:
def population_preset_for_years_ago(years_ago):
    target_bce=max(0.0,float(years_ago)-2026.0)
    anchor_map=defaults['population_anchors_bce']['anchors']
    years=np.array(sorted(int(k) for k in anchor_map),dtype=float)
    if target_bce<=years[0]: lo=hi=years[0]
    elif target_bce>=years[-1]: lo=hi=years[-1]
    else:
        hi_i=int(np.searchsorted(years,target_bce)); lo,hi=years[hi_i-1],years[hi_i]
    weight=0.0 if lo==hi else (target_bce-lo)/(hi-lo)
    out={}
    for name in regions:
        a=float(anchor_map[str(int(lo))][name]); b=float(anchor_map[str(int(hi))][name])
        out[name]=int(round(np.exp((1-weight)*np.log(a)+weight*np.log(b))))
    return target_bce,out,lo,hi

def apply_population_for_date(_=None):
    target_bce,values,lo,hi=population_preset_for_years_ago(founder_age.value)
    if auto_population.value:
        for name in regions: start_boxes[name].value=values[name]
    population_status.value=f'<b>Start preset:</b> ~{target_bce:,.0f} BCE, log-interpolated between {lo:,.0f} and {hi:,.0f} BCE anchors. Tasmania is an explicit teaching placeholder, not an 11 ka estimate.'

def collect_offdiag():
    off=np.zeros((len(regions),len(regions)),dtype=float)
    for (i,j),box in matrix_boxes.items(): off[i,j]=(box.value/100.0)*migration_scale.value
    return off

def barrier_args():
    if not tasmania_barrier.value: return [], None
    pair=(regions.index('Australia/Oceania'),regions.index('Tasmania'))
    release=None if permanent_barrier.value else float(tasmania_release.value)
    return [pair],release

def current_config():
    pairs,release=barrier_args()
    return dict(
        initial_population=[start_boxes[n].value for n in regions],
        target_population=[target_boxes[n].value for n in regions],
        parent_source_matrix=parent_source_matrix_from_offdiag(collect_offdiag()),
        mixing_strength=[mix_boxes[n].value for n in regions],
        founder_age_years=float(founder_age.value),generation_interval_years=float(generation_interval.value),
        founder_region=regions.index(founder_region.value),founder_pair_joint_children=int(joint_children.value),
        late_contact_age_years=float(late_contact_age.value),late_contact_multiplier=float(late_multiplier.value),
        region_names=regions,barrier_pairs=pairs,barrier_release_age_years=release,
    )

def apply_preset(name=None):
    spec=defaults['scenario_presets'][name or preset.value]
    migration_scale.value=float(spec['migration_scale']); late_multiplier.value=float(spec['late_contact_multiplier'])
    mix_scale=float(spec['mixing_scale'])
    for region,base in zip(regions,defaults['mixing_strength_default']['values']): mix_boxes[region].value=min(1.0,float(base)*mix_scale)
    tasmania_barrier.value=bool(spec['tasmania_barrier'])
    if spec['tasmania_release_years_ago'] is None: permanent_barrier.value=True
    else:
        permanent_barrier.value=False; tasmania_release.value=int(spec['tasmania_release_years_ago'])

founder_age.observe(lambda change: apply_population_for_date() if auto_population.value else None,names='value')
auto_population.observe(lambda change: apply_population_for_date() if change['new'] else None,names='value')
preset.observe(lambda change: apply_preset(change['new']),names='value')
apply_population_for_date(); apply_preset('Teaching default')


In [ ]:
scenario_panel=widgets.VBox([preset,founder_age,generation_interval,founder_region,joint_children,mode,replicates,seed,map_metric])
population_panel=widgets.VBox([auto_population,population_status,pop_grid])
mix_panel=widgets.VBox([migration_scale,late_contact_age,late_multiplier]+[mix_boxes[n] for n in regions])
barrier_panel=widgets.VBox([tasmania_barrier,permanent_barrier,tasmania_release,widgets.HTML('<small>The default hard barrier encodes Tasmania\'s separation before an 11 ka founder date. The post-contact reproductive rate is still user-selected.</small>')])
accordion=widgets.Accordion(children=[scenario_panel,population_panel,mix_panel,matrix_grid,barrier_panel],titles=('Scenario','Population','Migration & mixing','Parental-source matrix','Tasmania barrier'))
accordion.selected_index=0
display(accordion)


## Run, compare, and export

Save any parameterization as **Scenario A** or **Scenario B**, then compare their present-day regional outcomes side by side.

In [ ]:
run_button=widgets.Button(description='Run simulation',button_style='success',icon='play')
save_a=widgets.Button(description='Save as A',button_style='info')
save_b=widgets.Button(description='Save as B',button_style='warning')
compare_button=widgets.Button(description='Compare A vs B',icon='exchange')
export_button=widgets.Button(description='Export config JSON',icon='download')
output=widgets.Output(); compare_output=widgets.Output()
display(widgets.HBox([run_button,save_a,save_b,compare_button,export_button]),output,compare_output)


In [ ]:
def fmt_pct(x,d=2): return f'{100*x:.{d}f}%'

def summary_cards(det,stoch=None):
    diag=diagnose_scenario(det)
    cards=[('Global both-founder genealogy',fmt_pct(det.global_both_founders_fraction[-1])),('Mean founder DNA',fmt_pct(det.global_genetic_ancestry[-1],6)),('Limiting region',f'{diag.limiting_region}: {fmt_pct(diag.limiting_region_fraction)}')]
    if stoch is not None:
        cards += [('P(all regions ≥99%)',fmt_pct(stoch.probability_all_regions_above(.99))),('P(founder lineage extinct)',fmt_pct(stoch.any_founder_extinction_probability))]
    html='<div style="display:flex;gap:12px;flex-wrap:wrap;margin:8px 0 16px 0">'
    for title,value in cards:
        html += f'<div style="padding:12px 16px;border:1px solid #dfe5ec;border-radius:12px;background:#f8fafc;min-width:190px"><div style="font-size:12px;color:#5f6b7a">{title}</div><div style="font-size:22px;font-weight:700;margin-top:4px">{value}</div></div>'
    html+='</div>'
    display(HTML(html))
    return diag

def make_map(det,metric):
    if metric=='both': values=det.both_founders_fraction; title='Descended from both founders'
    elif metric=='any': values=det.any_founder_fraction; title='Descended from either founder'
    else: values=det.genetic_ancestry; title='Mean founder-pair autosomal ancestry'
    max_pop=float(det.populations.max()); base_matrix=current_config()['parent_source_matrix']
    edge_traces=[]
    for i in range(len(regions)):
        for j in range(i+1,len(regions)):
            rate=max(base_matrix[i,j],base_matrix[j,i])
            if rate>0:
                edge_traces.append(go.Scattergeo(lat=[coords[i,0],coords[j,0]],lon=[coords[i,1],coords[j,1]],mode='lines',line=dict(width=max(.5,450*rate),color='rgba(80,93,110,.28)'),hoverinfo='skip',showlegend=False))
    tas_i=regions.index('Tasmania'); aus_i=regions.index('Australia/Oceania')
    if tasmania_barrier.value:
        edge_traces.append(go.Scattergeo(lat=[coords[aus_i,0],coords[tas_i,0]],lon=[coords[aus_i,1],coords[tas_i,1]],mode='lines',line=dict(width=3,color='rgba(190,20,35,.75)',dash='dot'),hovertext='Tasmania hard barrier before release',hoverinfo='text',showlegend=False))
    def nodes(k):
        pop=det.populations[k]; size=11+38*np.sqrt(pop/max_pop)
        hover=[f'<b>{regions[i]}</b><br>Population: {pop[i]:,.0f}<br>Both: {fmt_pct(det.both_founders_fraction[k,i])}<br>Either: {fmt_pct(det.any_founder_fraction[k,i])}<br>Mean DNA: {fmt_pct(det.genetic_ancestry[k,i],7)}' for i in range(len(regions))]
        return go.Scattergeo(lat=coords[:,0],lon=coords[:,1],mode='markers+text',text=regions,textposition='bottom center',hovertext=hover,hoverinfo='text',marker=dict(size=size,color=100*values[k],cmin=0,cmax=100,colorscale='Turbo',showscale=True,colorbar=dict(title='%'),line=dict(width=1.2,color='white')),name=title)
    stride=max(1,len(det.generations)//55); ids=list(range(0,len(det.generations),stride))
    if ids[-1]!=len(det.generations)-1: ids.append(len(det.generations)-1)
    frames=[go.Frame(data=[nodes(k)],traces=[len(edge_traces)],name=str(k)) for k in ids]
    fig=go.Figure(data=edge_traces+[nodes(ids[0])],frames=frames)
    steps=[]
    for k in ids:
        yrs=int(round(det.years_before_present[k])); label='present' if yrs==0 else f'{yrs:,} y ago'
        steps.append(dict(method='animate',args=[[str(k)],dict(mode='immediate',frame=dict(duration=160,redraw=True),transition=dict(duration=0))],label=label))
    fig.update_layout(template='plotly_white',title=dict(text=title,x=.5),geo=dict(projection_type='natural earth',showland=True,landcolor='#eef1e8',showocean=True,oceancolor='#dcecf5',showcountries=True,countrycolor='white',bgcolor='white'),height=650,margin=dict(l=10,r=10,t=65,b=10),updatemenus=[dict(type='buttons',direction='left',x=.02,y=.02,buttons=[dict(label='▶ Play',method='animate',args=[None,dict(frame=dict(duration=160,redraw=True),fromcurrent=True,transition=dict(duration=0))]),dict(label='❚❚ Pause',method='animate',args=[[None],dict(mode='immediate',frame=dict(duration=0,redraw=False),transition=dict(duration=0))])])],sliders=[dict(active=0,currentvalue=dict(prefix='Time: '),pad=dict(t=28),steps=steps)])
    return fig

def make_dashboard(det,stoch=None):
    fig=make_subplots(rows=2,cols=2,specs=[[{'type':'xy'},{'type':'xy'}],[{'type':'xy'},{'type':'xy'}]],subplot_titles=('Regional both-founder genealogy','Global both-founder genealogy','Present regional genealogy vs DNA','Modeled regional population'))
    x=det.years_before_present
    for i,name in enumerate(regions):
        fig.add_trace(go.Scatter(x=x,y=100*det.both_founders_fraction[:,i],mode='lines',name=name,line=dict(width=2,color=region_colors[i]),legendgroup=name),row=1,col=1)
    fig.add_trace(go.Scatter(x=x,y=100*det.global_both_founders_fraction,mode='lines',name='Deterministic global',line=dict(width=3,color='#111827'),showlegend=False),row=1,col=2)
    if stoch is not None:
        lo=100*stoch.global_quantile(.05); med=100*stoch.global_quantile(.5); hi=100*stoch.global_quantile(.95)
        fig.add_trace(go.Scatter(x=x,y=lo,mode='lines',line=dict(width=0),showlegend=False,hoverinfo='skip'),row=1,col=2)
        fig.add_trace(go.Scatter(x=x,y=hi,mode='lines',fill='tonexty',fillcolor='rgba(45,125,210,.18)',line=dict(width=0),name='MC 5–95%',showlegend=False),row=1,col=2)
        fig.add_trace(go.Scatter(x=x,y=med,mode='lines',line=dict(width=2,color='#2d7dd2',dash='dash'),name='MC median',showlegend=False),row=1,col=2)
    fig.add_trace(go.Bar(y=regions,x=100*det.both_founders_fraction[-1],orientation='h',name='Both founders',marker_color='#d1495b',showlegend=False),row=2,col=1)
    fig.add_trace(go.Bar(y=regions,x=100*det.genetic_ancestry[-1],orientation='h',name='Mean DNA',marker_color='#00798c',showlegend=False),row=2,col=1)
    for i,name in enumerate(regions):
        fig.add_trace(go.Scatter(x=x,y=det.populations[:,i],mode='lines',name=name,line=dict(width=2,color=region_colors[i]),legendgroup=name,showlegend=False),row=2,col=2)
    fig.update_xaxes(autorange='reversed',title_text='Years before present',row=1,col=1); fig.update_xaxes(autorange='reversed',title_text='Years before present',row=1,col=2); fig.update_xaxes(title_text='Percent',row=2,col=1); fig.update_xaxes(autorange='reversed',title_text='Years before present',row=2,col=2)
    fig.update_yaxes(title_text='Percent',range=[0,100],row=1,col=1); fig.update_yaxes(title_text='Percent',range=[0,100],row=1,col=2); fig.update_yaxes(type='log',title_text='Population (log)',row=2,col=2)
    fig.update_layout(template='plotly_white',height=900,barmode='group',legend=dict(orientation='h',y=1.08,x=.5,xanchor='center'),margin=dict(l=50,r=30,t=100,b=40))
    return fig

def make_matrix_heatmap(matrix):
    z=100*matrix.copy(); np.fill_diagonal(z,np.nan)
    fig=go.Figure(go.Heatmap(z=z,x=regions,y=regions,colorscale='Blues',colorbar=dict(title='% parents'),hovertemplate='Destination: %{y}<br>Source: %{x}<br>%{z:.4f}% per parental draw<extra></extra>'))
    fig.update_layout(template='plotly_white',title='Parental-source matrix',xaxis_title='Source region',yaxis_title='Destination region',height=550,margin=dict(l=120,r=30,t=70,b=80))
    return fig

def run_current(show=True):
    cfg=current_config(); det=simulate_continental_explorer(**cfg)
    stoch=None
    if mode.value=='mc': stoch=simulate_continental_stochastic(**cfg,replicates=replicates.value,seed=seed.value)
    if show:
        diag=summary_cards(det,stoch)
        display(HTML('<h4>Automatic diagnosis</h4><ul>'+''.join(f'<li>{m}</li>' for m in diag.messages)+'</ul>'))
        if stoch is not None:
            rows=[]
            for t in [.5,.9,.99]: rows.append(f'<li>P(all 7 regions ≥ {int(100*t)}% both-founder descent) = <b>{fmt_pct(stoch.probability_all_regions_above(t))}</b></li>')
            display(HTML('<h4>Monte Carlo outcomes</h4><ul>'+''.join(rows)+f'<li>P(any-founder lineage extinct globally) = <b>{fmt_pct(stoch.any_founder_extinction_probability)}</b></li></ul><small>These probabilities are conditional on the selected parameters; they are not historical posterior probabilities.</small>'))
        make_map(det,map_metric.value).show(); make_dashboard(det,stoch).show(); make_matrix_heatmap(cfg['parent_source_matrix']).show()
    return cfg,det,stoch

def on_run(_):
    with output:
        clear_output(wait=True)
        try: run_current(True)
        except Exception as exc: print('Simulation error:',exc)

def save_scenario(label):
    cfg,det,stoch=run_current(False); saved_scenarios[label]=(cfg,det,stoch)
    return det

def on_save_a(_):
    with compare_output:
        det=save_scenario('A'); print(f'Saved A: global both-founder = {fmt_pct(det.global_both_founders_fraction[-1])}')
def on_save_b(_):
    with compare_output:
        det=save_scenario('B'); print(f'Saved B: global both-founder = {fmt_pct(det.global_both_founders_fraction[-1])}')
def on_compare(_):
    with compare_output:
        clear_output(wait=True)
        if 'A' not in saved_scenarios or 'B' not in saved_scenarios: print('Save both A and B first.'); return
        a=saved_scenarios['A'][1]; b=saved_scenarios['B'][1]; cmp=compare_scenarios(a,b)
        fig=go.Figure()
        fig.add_trace(go.Bar(y=regions,x=100*cmp['present_both_a'],orientation='h',name='Scenario A'))
        fig.add_trace(go.Bar(y=regions,x=100*cmp['present_both_b'],orientation='h',name='Scenario B'))
        fig.update_layout(template='plotly_white',barmode='group',title='A/B comparison: present both-founder genealogy',xaxis_title='Percent',height=520)
        fig.show()
        print('Global both-founder:', 'A',fmt_pct(cmp['global_both_a']),'| B',fmt_pct(cmp['global_both_b']))

def serializable_config():
    cfg=current_config(); out=dict(cfg); out['parent_source_matrix']=np.asarray(out['parent_source_matrix']).tolist(); out['barrier_pairs']=[list(x) for x in out['barrier_pairs']]; return out
def on_export(_):
    path=Path('continental_explorer_scenario.json'); path.write_text(json.dumps(serializable_config(),indent=2))
    if 'google.colab' in sys.modules:
        from google.colab import files; files.download(str(path))
    else: print('Wrote',path.resolve())

run_button.on_click(on_run); save_a.on_click(on_save_a); save_b.on_click(on_save_b); compare_button.on_click(on_compare); export_button.on_click(on_export)
on_run(None)


## Reading the result correctly

- **Hard barrier closed:** ancestry cannot cross that edge, regardless of elapsed time.
- **Monte Carlo probability:** stochastic probability conditional on your fixed parameters, not a posterior probability of history.
- **A/B comparison:** useful for asking whether a conclusion depends mainly on migration, endogamy, barrier timing, or founder survival.
- **Genealogy vs DNA:** near-universal genealogy can coexist with a tiny mean autosomal contribution.
- **Tasmania:** its population default is intentionally labeled a teaching placeholder because the repository does not contain a defensible 11 ka Tasmania population estimate.